# Chapter 8: AlphaGo and AlphaZero
### **RL: The Seminal Papers** by Rahul Shirale

This notebook trains an AlphaZero agent from random weights, by self-play, on a
board small enough that we know the right answer.

That last part is unusual and it is the point. Van den Herik, Uiterwijk and van
Rijswijck (2002) published the game-theoretic value of every small Connect Four
board: 6x6 is a **proven draw**, 6x7 is a **proven first-player win**, and 6x6
Othello is a rare **second-player** win. So when this agent finishes, we do not
have to squint at a loss curve and hope. We can ask whether it has found the
answer, and get a straight no.

**What you will see.** Loss falls monotonically. The agent beats a random
player 40-0-0 within two iterations and holds better than 0.9 against a depth-3
exact search. And it draws with itself something like 6% to 16% of the time,
where perfect play on this board would draw 100%. That gap is the most useful
number here: it is what tells you the run is *unfinished* rather than done, and
it is only visible because the board has a proven value.


## 1. What runs here, and what does not

The chapter has two halves and only one of them is runnable, which is worth
saying plainly rather than pretending otherwise.

**AlphaGo (2016)** is a 19x19 system: a 48-plane board encoding, a 13-layer
policy network trained on 30 million positions from the KGS server, a separate
value network that took 50 GPUs a week, and a fast rollout policy mixed into
every leaf evaluation. Nobody trains that on a laptop. Section 14 below runs
the one AlphaGo experiment that needs no dataset at all -- the timing
measurement that explains why the rollout policy existed in the first place.

**AlphaZero (2018)** is what is left after you delete the human games, the
hand-built features, the rollouts and the second network. That is small enough
to build from nothing here, at a smaller board size, and the algorithm is
unchanged -- only the budget is.

Every code cell in sections 2 through 13 is a file from
`src/part_2_methods/ch08_alphazero/`, verbatim. A test in the repository
compares them character by character, so what you run here and what you read
there cannot drift apart.


In [ ]:
# Colab has torch and matplotlib already. Locally: make install-full
# !pip install -q torch numpy matplotlib

import os
import time

import numpy as np
import torch

# The repository's test suite sets CH8_FAST=1 to execute this notebook end to
# end in seconds. It shrinks the board, the fleet and the search depth; it
# changes nothing about the algorithm. Leave it unset to run the chapter.
FAST = bool(os.environ.get("CH8_FAST"))

SEED = 0
torch.manual_seed(SEED)
RNG = np.random.default_rng(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print("torch  ", torch.__version__)
print("device ", DEVICE)


## 2. The contract

AlphaZero's headline claim is that one algorithm, with one set of
hyperparameters, masters several games. That claim only means something if the
agent talks to each game through an interface narrow enough that swapping the
game cannot smuggle knowledge in.

So the first file is not an algorithm. It is the seam. Everything below it is
rules; everything above it is the agent, and nothing above it imports a game.

The AlphaGo listings are the instructive counterexample. They call
`state.take_action()`, `state.get_liberties()` and `rollout.simulate()` --
methods on an interface that is never written down, because a system built for
one game never has to say what a game *is*.

One convention does all the sign work: **positions are canonical**. The side to
move is always `+1`, and `apply_move` negates the board on the way out. The
agent never asks whose turn it is, which deletes a whole category of bug.


In [ ]:
"""The contract every domain in this chapter satisfies.

AlphaZero's headline claim is that one algorithm, with one set of
hyperparameters, masters several games. That claim only means something if the
agent talks to each game through the same narrow interface. Everything below
the interface is game rules; everything above it is the agent. Nothing in
``mcts.py``, ``selfplay.py`` or ``train.py`` imports a game module -- they
import this one, which is what lets the same agent play Connect Four and
Othello without a line changing.

The AlphaGo listings in ``alphago.py`` are the counterexample. They call
``state.take_action()``, ``state.get_liberties()`` and ``rollout.simulate()``
on an interface that is never written down anywhere in the paper, because a
Go-only system never has to say what a game *is*.

Positions are always canonical: the side to move is +1 and its opponent is -1,
so the agent never needs to know whose turn it is. ``apply_move`` negates the
board on the way out, which removes an entire category of sign bug.
"""

from abc import ABC, abstractmethod

import numpy as np


class Game(ABC):
    """A two-player, zero-sum, perfect-information game."""

    name = "game"

    @property
    @abstractmethod
    def action_size(self):
        """Number of distinct actions, including a pass if the game has one."""

    @property
    @abstractmethod
    def board_shape(self):
        """Spatial shape of one plane, as (rows, cols)."""

    @property
    def input_planes(self):
        """Feature planes fed to the network: own stones, theirs, and a
        constant plane that gives the convolutions a sense of the edge."""
        return 3

    @abstractmethod
    def initial_board(self):
        """The opening position, from the first player's point of view."""

    @abstractmethod
    def legal_moves(self, board):
        """A 0/1 mask over actions, from the current player's point of view."""

    @abstractmethod
    def apply_move(self, board, action):
        """Play action and return the position with the sides swapped, so the
        result is again canonical for whoever moves next."""

    @abstractmethod
    def outcome(self, board):
        """None while the game is live, otherwise the result for the player to
        move: +1 win, -1 loss, 0 draw."""

    def solved_value(self):
        """The proven game-theoretic value of the opening position, or None.

        This is what makes the project's convergence measurable rather than
        merely plausible. Van den Herik, Uiterwijk and van Rijswijck (2002)
        catalogue the small boards; a subclass returns their entry.
        """
        return None

    def encode(self, board):
        """Stack the canonical board into planes the network can read."""
        own = (board == 1).astype(np.float32)
        theirs = (board == -1).astype(np.float32)
        ones = np.ones_like(own)
        return np.stack([own, theirs, ones])

    def key(self, board):
        """A hashable position key, for transposition tables and solvers."""
        return board.tobytes()

    def symmetries(self, board, pi):
        """Equivalent (position, policy) pairs, used to multiply training data.

        AlphaGo exploited the eight reflections and rotations of the Go board.
        Most games have fewer; the default is to claim none.
        """
        return [(board, pi)]

    def render(self, board):
        return str(board)


## 3. Connect Four, and why the board is 6x6

The rules are about forty lines. That ratio matters: this is a chapter about an
algorithm, and a domain whose rules engine is longer than the agent teaches you
about the domain instead. Small-board Go was the obvious alternative and was
rejected for exactly that reason -- captures, ko, suicide and area scoring are
several hundred lines that say nothing about reinforcement learning.

`_build_lines` is worth a second look. Precomputing every window of four as flat
indices turns the terminal test into a list comparison. The first version of
that test cost 87 microseconds per call, which is fatal when one self-play
iteration asks the question millions of times; plain-Python tuples brought it to
5.1 microseconds, *five times faster than the NumPy version*, because these
arrays are far too small to repay NumPy's per-call overhead.


In [ ]:
"""Connect Four, on the standard board and on the smaller solved ones.

Van den Herik, Uiterwijk and van Rijswijck (2002) catalogue the
game-theoretic value of every small board: 4x4, 6x4, 4x5, 6x5, 4x6, 6x6 and
4x7 are draws, and the standard 6x7 board is a first-player win. Those values
are the ground truth this chapter checks a trained agent against, and they are
why the project's default board is 6x6 rather than 6x7 -- on a drawn board an
agent approaching perfect play draws against itself every time, which is a
convergence signal a network cannot fake by learning to output zero.
"""

import numpy as np


#: Board (rows x cols) -> game-theoretic value for the first player, from
#: van den Herik, Uiterwijk and van Rijswijck (2002), "Games solved: Now and
#: in the future", section 3.1.1.
#:
#: Named for its game rather than called SOLVED_VALUES because the notebook
#: inlines both this module and othello.py into one namespace, where two
#: constants of the same name silently become one -- and Connect Four would
#: start reporting Othello's answers.
CONNECT_FOUR_SOLVED_VALUES = {
    (4, 4): 0,
    (6, 4): 0,
    (4, 5): 0,
    (6, 5): 0,
    (4, 6): 0,
    (6, 6): 0,
    (4, 7): 0,
    (6, 7): 1,
}


class ConnectFour(Game):
    def __init__(self, rows=6, cols=7, win_length=4):
        self.rows = rows
        self.cols = cols
        self.win_length = win_length
        self.name = f"connect_four_{rows}x{cols}"
        self._lines = self._build_lines()

    @property
    def action_size(self):
        return self.cols

    @property
    def board_shape(self):
        return (self.rows, self.cols)

    def initial_board(self):
        return np.zeros((self.rows, self.cols), dtype=np.int8)

    def legal_moves(self, board):
        # A column is playable while its top cell is empty.
        return (board[0] == 0).astype(np.int8)

    def apply_move(self, board, action):
        board = board.copy()
        # Gravity: the disc falls to the lowest empty cell in the column.
        rows = np.nonzero(board[:, action] == 0)[0]
        if rows.size == 0:
            raise ValueError(f"column {action} is full")
        board[rows[-1], action] = 1
        # Hand the position to the opponent, who now reads as +1.
        return -board

    def outcome(self, board):
        # outcome() is asked after a move has been played, so a completed line
        # of -1 belongs to the player who just moved -- which is a loss for the
        # player to move now, hence the sign convention holding without any
        # special case here.
        cells = board.ravel().tolist()
        for line in self._lines:
            first = cells[line[0]]
            if first and all(cells[i] == first for i in line[1:]):
                return int(first)
        if 0 not in cells:
            return 0
        return None

    def _build_lines(self):
        """Every window of win_length cells, as flat indices.

        Precomputing these once is what makes the terminal test affordable. It
        matters more than it looks: one self-play iteration asks whether a
        position is terminal millions of times. The first implementation of
        this test cost 87 microseconds a call; plain-Python index tuples
        brought it to 5.1 -- five times faster than the NumPy version, because
        these arrays are far too small to repay NumPy's per-call overhead.
        """
        lines = []
        for row in range(self.rows):
            for col in range(self.cols):
                for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
                    end_r = row + dr * (self.win_length - 1)
                    end_c = col + dc * (self.win_length - 1)
                    if not (0 <= end_r < self.rows and 0 <= end_c < self.cols):
                        continue
                    lines.append([
                        (row + dr * k) * self.cols + (col + dc * k)
                        for k in range(self.win_length)
                    ])
        return [tuple(line) for line in lines]

    def symmetries(self, board, pi):
        # Mirroring the columns leaves Connect Four unchanged, which doubles
        # every training example for free. This is AlphaGo's eight-fold board
        # symmetry trick, on a board that only has two.
        return [(board, pi), (board[:, ::-1].copy(), pi[::-1].copy())]

    def solved_value(self):
        """The proven value of the opening position, or None if unknown."""
        return CONNECT_FOUR_SOLVED_VALUES.get((self.rows, self.cols))

    def render(self, board):
        glyph = {0: ".", 1: "X", -1: "O"}
        lines = ["".join(glyph[int(v)] for v in row) for row in board]
        lines.append("".join(str(c % 10) for c in range(self.cols)))
        return "\n".join(lines)


## 4. A second game, for the claim that actually matters

The chapter asserts AlphaZero's generality by naming chess and shogi. We cannot
run those. We can run a second game through the same interface with **not one
line of the agent changing**, which is the same claim at a scale that finishes
over lunch.

Othello is the closer cousin to Go: stones on a square grid, captures by
surrounding, a pass when you are stuck, and a territory count at the end. It
also has the eight symmetries of the square, which is AlphaGo's data-multiplying
trick available here for the cost of a few transposes.

And 6x6 Othello is solved, with the interesting answer: it is a **second-player
win**. An agent that comes out of self-play convinced that moving first is an
advantage here has learned something false, and we can prove it.


In [ ]:
"""Othello on a 6x6 board -- the second domain, and the point of the interface.

Othello is the closer cousin to Go of the two domains here: stones sit on a
square grid, a move captures enemy stones by surrounding them, a player with
no move must pass, and the game is decided by counting territory at the end.
It is also solved at this size, and the answer is the interesting one -- van
den Herik and colleagues (2002) record 6x6 Othello as a rare genuine
*second*-player win, solved by Feinstein. An agent that comes out of self-play
convinced that moving first is an advantage here has learned something false,
and unlike almost anywhere else in reinforcement learning, we can prove it.

Nothing in the agent changes to play this. That is the whole argument: the
chapter asserts AlphaZero's generality by naming chess and shogi, and this
file lets a reader watch the same claim at a scale a laptop can finish.
"""

import numpy as np


DIRECTIONS = [(-1, -1), (-1, 0), (-1, 1),
              (0, -1),           (0, 1),
              (1, -1),  (1, 0),  (1, 1)]

#: First-player value of the opening position, from van den Herik, Uiterwijk
#: and van Rijswijck (2002), section 3.4. Second player wins at 6x6. Named for
#: its game for the reason connect_four.py records.
OTHELLO_SOLVED_VALUES = {(6, 6): -1}


class Othello(Game):
    def __init__(self, size=6):
        if size % 2:
            raise ValueError("Othello needs an even board")
        self.size = size
        self.name = f"othello_{size}x{size}"

    @property
    def action_size(self):
        # One action per square, plus an explicit pass.
        return self.size * self.size + 1

    @property
    def board_shape(self):
        return (self.size, self.size)

    @property
    def pass_action(self):
        return self.size * self.size

    def initial_board(self):
        board = np.zeros((self.size, self.size), dtype=np.int8)
        mid = self.size // 2
        board[mid - 1, mid - 1] = -1
        board[mid, mid] = -1
        board[mid - 1, mid] = 1
        board[mid, mid - 1] = 1
        return board

    def _flips(self, board, row, col):
        """Discs the current player would turn by playing at (row, col)."""
        if board[row, col] != 0:
            return []
        captured = []
        for dr, dc in DIRECTIONS:
            run = []
            r, c = row + dr, col + dc
            while 0 <= r < self.size and 0 <= c < self.size and board[r, c] == -1:
                run.append((r, c))
                r, c = r + dr, c + dc
            # The run only counts if our own disc closes the sandwich.
            if run and 0 <= r < self.size and 0 <= c < self.size and board[r, c] == 1:
                captured.extend(run)
        return captured

    def legal_moves(self, board):
        mask = np.zeros(self.action_size, dtype=np.int8)
        for row in range(self.size):
            for col in range(self.size):
                if self._flips(board, row, col):
                    mask[row * self.size + col] = 1
        if not mask.any():
            # With nothing to play, passing is the only legal action.
            mask[self.pass_action] = 1
        return mask

    def apply_move(self, board, action):
        board = board.copy()
        if action != self.pass_action:
            row, col = divmod(action, self.size)
            captured = self._flips(board, row, col)
            if not captured:
                raise ValueError(f"square {(row, col)} flips nothing")
            board[row, col] = 1
            for r, c in captured:
                board[r, c] = 1
        return -board

    def outcome(self, board):
        # The game is live while either side has a real move. Two passes in a
        # row end it, which is why both boards are tested rather than one.
        if self._has_move(board) or self._has_move(-board):
            return None
        mine = int((board == 1).sum())
        theirs = int((board == -1).sum())
        if mine > theirs:
            return 1
        if theirs > mine:
            return -1
        return 0

    def _has_move(self, board):
        for row in range(self.size):
            for col in range(self.size):
                if self._flips(board, row, col):
                    return True
        return False

    def symmetries(self, board, pi):
        """Othello's board has the eight symmetries of the square.

        This is the same trick AlphaGo used on the Go board, and it is worth
        eight times the training data for the cost of a few transposes. The
        pass action sits outside the grid, so it is sliced off, the square part
        is rotated, and it is put back unchanged.
        """
        out = []
        grid = pi[:-1].reshape(self.size, self.size)
        for rot in range(4):
            b = np.rot90(board, rot)
            p = np.rot90(grid, rot)
            for flip in (False, True):
                bb = np.fliplr(b) if flip else b
                pp = np.fliplr(p) if flip else p
                policy = np.concatenate([pp.ravel(), pi[-1:]])
                out.append((np.ascontiguousarray(bb),
                            np.ascontiguousarray(policy)))
        return out

    def solved_value(self):
        return OTHELLO_SOLVED_VALUES.get((self.size, self.size))

    def render(self, board):
        glyph = {0: ".", 1: "X", -1: "O"}
        return "\n".join("".join(glyph[int(v)] for v in row) for row in board)


## 5. The oracle

Alpha-beta with a transposition table settles the smallest boards outright. This
is the same brute force Allen used to solve Connect Four in 1988, and here it
stands for the ground truth a self-play agent has to rediscover with none of it.

The flag stored beside each cached value is not a detail. A value produced under
a narrow alpha-beta window is not exact, and caching it as though it were makes
the solver quietly wrong on later lookups -- which defeats the entire purpose of
having an oracle.


In [ ]:
"""An exact solver -- the oracle the trained agent is checked against.

Almost no reinforcement learning example has a correct answer to compare
against. This one does, because van den Herik, Uiterwijk and van Rijswijck
(2002) published the game-theoretic value of every small Connect Four board
and of 6x6 Othello, and because alpha-beta with a transposition table can
reprove the smallest of them here in seconds rather than taking them on
trust.

That oracle is what turns "the loss went down" into a real measurement. An
agent measured only by its training curve looks finished; an agent measured
against a proven draw is visibly nowhere near it.

Run it:

    python -m src.part_2_methods.ch08_alphazero.solver
"""

import argparse
import time

import numpy as np

EXACT, LOWER, UPPER = 0, 1, 2


def solve(game, board, alpha=-1.0, beta=1.0, table=None):
    """Exact value of a position for the player to move.

    The transposition table stores a flag alongside each value. Without it, a
    value produced under a narrow alpha-beta window would be cached as though
    it were exact, and the solver would quietly return wrong answers on later
    lookups -- which defeats the entire point of an oracle.

    Only tractable on the small boards. The standard 6x7 board has on the
    order of 10^14 positions and is far out of reach of pure Python.
    """
    if table is None:
        table = {}
    result = game.outcome(board)
    if result is not None:
        return float(result)

    key = game.key(board)
    cached = table.get(key)
    if cached is not None:
        value, flag = cached
        if flag == EXACT:
            return value
        if flag == LOWER and value >= beta:
            return value
        if flag == UPPER and value <= alpha:
            return value

    original_alpha = alpha
    best = -1.0
    for action in np.nonzero(game.legal_moves(board))[0]:
        child = game.apply_move(board, int(action))
        # The child is canonical for the opponent, so its value negates.
        value = -solve(game, child, -beta, -alpha, table)
        best = max(best, value)
        alpha = max(alpha, value)
        if alpha >= beta:
            break

    if best <= original_alpha:
        flag = UPPER
    elif best >= beta:
        flag = LOWER
    else:
        flag = EXACT
    table[key] = (best, flag)
    return best


def negamax(game, board, depth, alpha=-1.0, beta=1.0, heuristic=None):
    """Depth-limited search, used as a fixed-strength baseline opponent.

    Unlike ``solve`` this is not exact, which is the point: it is a bar the
    agent can be measured against long before it is anywhere near perfect.
    """
    result = game.outcome(board)
    if result is not None:
        return float(result)
    if depth == 0:
        return heuristic(board) if heuristic else 0.0

    best = -2.0
    for action in np.nonzero(game.legal_moves(board))[0]:
        child = game.apply_move(board, int(action))
        value = -negamax(game, child, depth - 1, -beta, -alpha, heuristic)
        best = max(best, value)
        alpha = max(alpha, value)
        if alpha >= beta:
            break
    return best


def best_action(game, board, depth, heuristic=None, rng=None):
    """The move a depth-limited search prefers, ties broken at random.

    Ties are broken randomly on purpose. A deterministic baseline plays one
    game over and over, and a match against it measures a single line rather
    than a player.
    """
    scored = []
    for action in np.nonzero(game.legal_moves(board))[0]:
        child = game.apply_move(board, int(action))
        value = -negamax(game, child, depth - 1, -1.0, 1.0, heuristic)
        scored.append((value, int(action)))
    best = max(score for score, _ in scored)
    tied = [action for score, action in scored if score == best]
    if rng is not None:
        return int(rng.choice(tied))
    return tied[0]


def disc_difference(board):
    """A crude Othello heuristic: who holds more of the board, scaled to the
    [-1, 1] range the search works in."""
    mine = int((board == 1).sum())
    theirs = int((board == -1).sum())
    total = mine + theirs
    return (mine - theirs) / total if total else 0.0


#: Boards small enough to settle in seconds. The rest of the published table
#: is hours of pure Python, so the chapter cites it rather than recomputing.
DEFAULT_BOARDS = [(4, 4), (4, 5), (6, 4)]


def main(argv=None):
    """Reprove the small boards and check them against the published values."""

    parser = argparse.ArgumentParser(description=main.__doc__)
    parser.add_argument(
        "--boards", default=",".join(f"{r}x{c}" for r, c in DEFAULT_BOARDS),
        help="comma-separated RxC list, e.g. 4x4,4x5,6x4")
    args = parser.parse_args(argv)

    boards = []
    for spec in args.boards.split(","):
        rows, cols = spec.lower().split("x")
        boards.append((int(rows), int(cols)))

    print("Connect Four, solved exactly by alpha-beta with a transposition")
    print("table, against van den Herik, Uiterwijk and van Rijswijck (2002).")
    print()
    print(f"{'board':>7}  {'solved':>7}  {'published':>9}  "
          f"{'positions':>10}  {'seconds':>8}")

    verdicts = []
    for rows, cols in boards:
        game = ConnectFour(rows, cols)
        table = {}
        started = time.time()
        value = int(solve(game, game.initial_board(), table=table))
        elapsed = time.time() - started
        published = game.solved_value()
        agrees = published is not None and value == published
        verdicts.append(agrees)
        shown = "-" if published is None else f"{published:+d}"
        print(f"{rows}x{cols:<5}  {value:+7d}  {shown:>9}  "
              f"{len(table):10,}  {elapsed:8.1f}")

    print()
    if all(verdicts):
        print("every board matches the published value.")
    else:
        print("MISMATCH: a board disagrees with the published value.")
    return verdicts


if __name__ == "__main__":
    main()


In [ ]:
# Reprove the published values before trusting anything downstream.
# 4x4 is instant; 4x5 takes about ten seconds.
for rows, cols in ([(4, 4)] if FAST else [(4, 4), (4, 5)]):
    game = ConnectFour(rows, cols)
    table = {}
    started = time.time()
    value = int(solve(game, game.initial_board(), table=table))
    print(f"{rows}x{cols}  solved {value:+d}  published "
          f"{game.solved_value():+d}  "
          f"({len(table):,} positions, {time.time() - started:.1f}s)")


## 6. One trunk, two heads

AlphaGo trained a policy network and a value network with the same architecture
and no shared weights, which meant learning the same board features twice and
paying for two forward passes at every leaf. Merging them into one residual
trunk with two heads is most of what AlphaGo Zero changed.

The architecture below is the published one with two numbers turned down:
`blocks` from 19 to 4 and `channels` from 256 to 64. Nothing else differs. That
is a factor of fifty in parameters and the difference between thousands of TPUs
and your laptop -- a difference of budget, not of kind, which is the honest
version of "you can run AlphaZero at home".

Note that `forward` returns **raw logits**. A softmax here would spend
probability mass on moves the rules forbid; it happens later, in `_expand`,
where the legal mask is known.


In [ ]:
"""The unified AlphaZero network, sized for a board a laptop can train on.

The published network -- printed in ``alphago.py`` as the contrast -- is a
19-block, 256-channel residual tower with 22,837,864 parameters, trained on
thousands of TPUs. The architecture here is the same shape: a shared residual
trunk under a policy head and a value head. Two scale knobs are turned down,
``blocks`` from 19 to 4 and ``channels`` from 256 to 64, which brings it to
456,647 parameters on the 6x6 board and a full training run inside an hour.

Nothing about the algorithm changes. Only the budget does. That is the honest
version of "you can run AlphaZero at home", and it is worth being explicit
that the gap is a factor of fifty in parameters, not a difference in kind.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        # The skip connection lets gradients reach the early layers.
        return F.relu(out + x)


class AlphaZeroNet(nn.Module):
    """One trunk, two heads -- AlphaGo's two separate networks merged.

    AlphaGo trained a policy network and a value network with the same
    architecture and no shared weights, which meant learning the same board
    features twice. Merging them is most of what AlphaGo Zero changed, and it
    is why a leaf evaluation here costs one forward pass rather than two.
    """

    def __init__(self, input_planes, board_shape, action_size,
                 channels=64, blocks=4):
        super().__init__()
        rows, cols = board_shape
        self.stem = nn.Sequential(
            nn.Conv2d(input_planes, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
        )
        self.trunk = nn.Sequential(*[ResBlock(channels) for _ in range(blocks)])

        # Policy head: collapse to a couple of planes, then score every action.
        self.policy_conv = nn.Sequential(
            nn.Conv2d(channels, 32, 1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.policy_fc = nn.Linear(32 * rows * cols, action_size)

        # Value head: collapse to one plane, then to a single number.
        self.value_conv = nn.Sequential(
            nn.Conv2d(channels, 32, 1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.value_fc = nn.Sequential(
            nn.Linear(32 * rows * cols, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 1),
            nn.Tanh(),
        )

    def forward(self, x):
        x = self.trunk(self.stem(x))
        policy = self.policy_fc(self.policy_conv(x).flatten(1))
        value = self.value_fc(self.value_conv(x).flatten(1))
        # Raw logits: the softmax happens where the legal mask is known, in
        # BatchedMCTS._expand. A softmax here would spend probability mass on
        # moves the rules forbid.
        return policy, value.squeeze(-1)

    def parameter_count(self):
        return sum(p.numel() for p in self.parameters())


def build(game, channels=64, blocks=4):
    """A network shaped by the game, which is the only thing it reads."""
    return AlphaZeroNet(
        input_planes=game.input_planes,
        board_shape=game.board_shape,
        action_size=game.action_size,
        channels=channels,
        blocks=blocks,
    )


def pick_device(preference="auto"):
    """Whatever accelerator this laptop happens to have.

    MPS is chosen where it exists because the self-play fleet is batched: the
    thing that makes MPS a poor fit for chapter 7's token-by-token generation
    -- a fixed per-call latency of roughly 17 ms -- costs nothing here, where
    one call answers several hundred positions.
    """
    if preference != "auto":
        return torch.device(preference)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


In [ ]:
game = ConnectFour(4, 4) if FAST else ConnectFour(6, 6)
device = torch.device(DEVICE)
net = build(game, channels=64, blocks=4).to(device)
net.eval()

print(f"this project      {net.parameter_count():>12,} parameters")
print(f"AlphaGo Zero      {22_837_864:>12,}   (19 blocks x 256 channels)")
print(f"ratio             {22_837_864 / net.parameter_count():>12.0f}x")


## 7. PUCT, and the rollout AlphaZero deleted

Here is the one structural difference between the two searches.

AlphaGo evaluated a leaf **twice**: once with the value network, and once by
playing a fast random game to the end, mixing the two with a parameter lambda.
It did that because in 2016 the value network alone was not trustworthy enough,
and because the rollout policy was about fifteen hundred times faster than the
network (section 14 measures that gap). AlphaZero deletes the rollout entirely.
The value head is the only evaluator, a leaf costs exactly one forward pass, and
lambda stops existing.

Two implementation details carry more weight than they look like they should:

- **Child boards are lazy.** Expanding a node creates one child per legal move,
  but a search visits only a few of them. Materialising every child's board up
  front is the single most expensive thing a naive implementation does.
- **`max(1, parent.visit_count)`** is a cold-start guard. Without it, the very
  first simulation has `sqrt(0)` in the numerator, every child scores exactly
  zero, and the opening move is chosen by dictionary order rather than by the
  policy head's opinion.

And the search runs **several positions in lockstep**, so the leaves from every
game in the fleet cross the device boundary in one batch. That is the same idea
the chapter describes for AlphaGo's distributed search, doing real work here
instead of illustrating.


In [ ]:
"""PUCT search, evaluated by the network rather than by rollouts.

This is the one structural difference between AlphaGo's search and
AlphaZero's. AlphaGo evaluated a leaf twice -- once with the value network and
once by playing a fast random game to the end -- and mixed the two with the
parameter lambda, because in 2016 the value network alone was not trustworthy
enough and the rollout policy was fifteen hundred times faster than the
network (``benchmark.py`` reproduces that gap). AlphaZero deletes the rollout.
The value head is the only evaluator, so a leaf costs exactly one forward
pass, and lambda stops existing.

Searches for several positions are driven in lockstep so their leaves can be
evaluated in a single batch. That is the same trick the chapter describes for
AlphaGo's distributed search, doing real work here instead of illustrating:
the network is far too slow to call one position at a time, so the search
collects pending leaves across the whole fleet and sends them through
together.
"""

import math

import numpy as np
import torch


class Node:
    """A position in the tree, whose board is built only when it is needed.

    Expanding a node creates one child per legal move, but a search visits
    only a few of them. Materialising every child's board up front is the
    single most expensive thing a naive implementation does, so children
    remember the move that makes them and compute the board on first visit.
    """

    __slots__ = ("prior", "visit_count", "value_sum", "children", "_board",
                 "terminal_value", "parent", "action")

    def __init__(self, board=None, prior=0.0, parent=None, action=None):
        self._board = board
        self.prior = prior
        self.visit_count = 0
        self.value_sum = 0.0
        self.children = {}
        self.terminal_value = None
        self.parent = parent
        self.action = action

    def board(self, game):
        if self._board is None:
            self._board = game.apply_move(self.parent.board(game), self.action)
        return self._board

    @property
    def expanded(self):
        return bool(self.children)

    def q_value(self):
        if self.visit_count == 0:
            return 0.0
        return self.value_sum / self.visit_count


def puct_score(parent, child, c_puct):
    """Q + U, the rule that decides where the search spends its next visit."""
    # Unvisited children score on their prior alone, which is what makes the
    # policy head's opinion the thing that shapes the tree's first look.
    # max(1, ...) is the cold-start guard: on the very first simulation the
    # parent has no visits, sqrt(0) is zero, and every child would score
    # exactly zero -- making the choice of opening move arbitrary rather than
    # prior-driven.
    exploration = c_puct * child.prior * math.sqrt(
        max(1, parent.visit_count)) / (1 + child.visit_count)
    if child.visit_count == 0:
        return exploration
    # A child's Q is from the opponent's point of view, so it negates.
    return -child.q_value() + exploration


def select_child(node, c_puct):
    return max(node.children.items(),
               key=lambda item: puct_score(node, item[1], c_puct))


class BatchedMCTS:
    def __init__(self, game, net, device, c_puct=1.5,
                 dirichlet_alpha=0.6, dirichlet_weight=0.25):
        self.game = game
        self.net = net
        self.device = device
        self.c_puct = c_puct
        self.dirichlet_alpha = dirichlet_alpha
        self.dirichlet_weight = dirichlet_weight

    def _evaluate(self, boards):
        """One forward pass for every pending leaf in the batch."""
        if not boards:
            return [], []
        planes = np.stack([self.game.encode(b) for b in boards])
        tensor = torch.from_numpy(planes).to(self.device)
        with torch.no_grad():
            logits, values = self.net(tensor)
            priors = torch.softmax(logits.float(), dim=1).cpu().numpy()
        return priors, values.float().cpu().numpy()

    def _expand(self, node, priors):
        mask = self.game.legal_moves(node.board(self.game))
        masked = priors * mask
        total = masked.sum()
        # A network that puts all its mass on illegal moves still has to
        # produce a distribution, so fall back to uniform over legal moves.
        # Early in training this happens often enough to matter.
        masked = masked / total if total > 1e-8 else mask / mask.sum()
        for action in np.nonzero(mask)[0]:
            action = int(action)
            node.children[action] = Node(prior=float(masked[action]),
                                         parent=node, action=action)

    def search(self, boards, simulations, add_noise=True, rng=None,
               trace=None, roots=None):
        """Visit-count distributions for every position in ``boards``.

        Returns ``(policies, roots)``. The roots come back so the caller can
        keep the subtree under the move it plays -- see ``selfplay.py``.

        Pass a list as ``trace`` and the search appends the root distribution
        after every simulation, which is what ``animate.py`` plays back.
        """
        rng = rng or np.random.default_rng()
        if roots is None:
            roots = [Node(board) for board in boards]

        unexpanded = [r for r in roots if not r.expanded]
        if unexpanded:
            priors, _ = self._evaluate([r.board(self.game) for r in unexpanded])
            for root, prior in zip(unexpanded, priors):
                self._expand(root, prior)

        if add_noise:
            for root in roots:
                self._add_dirichlet_noise(root, rng)

        for _ in range(simulations):
            paths, leaves, pending = [], [], []
            for root in roots:
                path, leaf = self._descend(root)
                paths.append(path)
                leaves.append(leaf)
                if leaf.terminal_value is None:
                    pending.append(leaf)

            # The batch: every game in the fleet contributed at most one leaf,
            # and they all cross the device boundary once.
            priors, values = self._evaluate(
                [leaf.board(self.game) for leaf in pending])
            lookup = {}
            for index, leaf in enumerate(pending):
                self._expand(leaf, priors[index])
                lookup[id(leaf)] = float(values[index])

            for path, leaf in zip(paths, leaves):
                value = (leaf.terminal_value if leaf.terminal_value is not None
                         else lookup[id(leaf)])
                self._backup(path, value)

            if trace is not None:
                trace.append([self._visit_distribution(r) for r in roots])

        return [self._visit_distribution(root) for root in roots], roots

    def _descend(self, root):
        node = root
        path = [node]
        while node.expanded and node.terminal_value is None:
            _, node = select_child(node, self.c_puct)
            path.append(node)
        if node.terminal_value is None:
            result = self.game.outcome(node.board(self.game))
            if result is not None:
                node.terminal_value = float(result)
        return path, node

    def _backup(self, path, value):
        # Walk back to the root, flipping the sign at every level because what
        # is good for one player is bad for the other.
        for node in reversed(path):
            node.visit_count += 1
            node.value_sum += value
            value = -value

    def _add_dirichlet_noise(self, root, rng):
        """Noise at the root keeps self-play from replaying one line forever.

        Applied to the root only, and only once per search. Adding it again to
        a reused subtree would compound it every ply.
        """
        actions = list(root.children)
        noise = rng.dirichlet([self.dirichlet_alpha] * len(actions))
        for action, sample in zip(actions, noise):
            child = root.children[action]
            child.prior = ((1 - self.dirichlet_weight) * child.prior
                           + self.dirichlet_weight * sample)

    def _visit_distribution(self, root):
        counts = np.zeros(self.game.action_size, dtype=np.float32)
        for action, child in root.children.items():
            counts[action] = child.visit_count
        total = counts.sum()
        return counts / total if total else counts


## 8. Self-play as a fleet

A single game of Connect Four asks the network for a few thousand evaluations,
every one of them a single position. A laptop GPU answers a batch of 256
positions in about the same wall-clock time as a batch of one -- measured at
17.5 ms either way on Apple MPS -- so playing one game at a time leaves almost
all of the hardware idle. Here a few hundred games advance together and share
every forward pass.

The comment block at the bottom of `play_batch` is the most important dozen
lines in this notebook. The result a game reports belongs to the player to move
in the *terminal* position -- the one who did **not** make the last move. Walking
backwards from there, the sign flips at every ply. Get that backwards and
nothing looks wrong: training runs, the loss falls, and the agent patiently
learns to lose. There is a test in the repository for exactly this, because it
is the one failure here that a curve will never show you.


In [ ]:
"""Self-play, run as a fleet rather than one game at a time.

A single game of Connect Four asks the network for a few thousand
evaluations, all of them one position wide. A laptop GPU answers a batch of
256 positions in about the same wall-clock time as a batch of one -- measured
at 17.5 ms either way on Apple MPS -- so playing one game at a time wastes
almost all of the hardware. Here a few hundred games advance in lockstep and
their searches share every forward pass.

The other saving is tree reuse. Once a move is played, the subtree under the
chosen child is already built and already carries visit statistics, so it
becomes the next root rather than being thrown away and re-expanded.
"""

import numpy as np


class Example:
    """One stored position: what the search concluded, and how it turned out.

    ``player_value`` stays None until the game ends, because the label is the
    result *as the player to move in this position saw it*, and nobody knows
    that until there is a result.
    """

    __slots__ = ("board", "policy", "player_value")

    def __init__(self, board, policy):
        self.board = board
        self.policy = policy
        self.player_value = None


def play_batch(game, mcts, num_games, simulations, temperature_moves=8,
               rng=None, max_plies=None):
    """Play num_games in lockstep and return (examples, results).

    Every position is stored with the visit-count distribution the search
    produced -- the policy target, which is better than the raw network
    because the search improved on it -- and once the game ends each position
    is labelled with the result as seen by whoever was to move there.
    """
    rng = rng or np.random.default_rng()
    max_plies = max_plies or (game.board_shape[0] * game.board_shape[1] + 2)

    boards = [game.initial_board() for _ in range(num_games)]
    histories = [[] for _ in range(num_games)]
    finished = [None] * num_games
    live = list(range(num_games))
    ply = 0
    roots = None

    while live and ply < max_plies:
        policies, search_roots = mcts.search(
            [boards[i] for i in live], simulations, add_noise=True, rng=rng,
            roots=roots)

        still_live = []
        next_roots = []
        for slot, index in enumerate(live):
            policy = policies[slot]
            histories[index].append(Example(boards[index].copy(), policy))

            action = _choose(policy, ply, temperature_moves, rng)
            boards[index] = game.apply_move(boards[index], action)

            result = game.outcome(boards[index])
            if result is None:
                still_live.append(index)
                # Tree reuse: the chosen child becomes the next root. Detaching
                # the parent lets the rest of the old tree be collected, which
                # on a fleet of a few hundred games is the difference between
                # steady memory and none left.
                new_root = search_roots[slot].children[action]
                new_root.parent = None
                next_roots.append(new_root)
            else:
                # result is from the point of view of the player to move in the
                # new position, which is the player who did not just move.
                finished[index] = float(result)

        live = still_live
        roots = next_roots if live else None
        ply += 1

    for index in live:
        # Anything still running at the ply cap is scored as a draw.
        finished[index] = 0.0

    examples = []
    for index in range(num_games):
        final = finished[index]
        history = histories[index]
        # `final` is the result seen by the player to move in the terminal
        # position -- the one who did NOT make the last move. Walking back from
        # there the sign flips at every ply, so each stored position ends up
        # labelled with the result as its own mover saw it. Get this backwards
        # and training still runs, the loss still falls, and the agent learns
        # to lose; test_self_play_labels_the_winner_positively exists for it.
        for offset, example in enumerate(reversed(history)):
            example.player_value = -final if offset % 2 == 0 else final
        for example in history:
            for board, policy in game.symmetries(example.board, example.policy):
                examples.append((game.encode(board), policy,
                                 np.float32(example.player_value)))
    return examples, finished


def _choose(policy, ply, temperature_moves, rng):
    if ply < temperature_moves:
        # Early moves are sampled, so the fleet explores different openings
        # instead of grinding the same line a few hundred times. This is
        # AlphaZero's temperature schedule, at its two extreme settings.
        return int(rng.choice(len(policy), p=policy))
    return int(np.argmax(policy))


## 9. Measuring it against something that is not itself

Training loss falls whether or not the agent is learning anything useful, so the
loop needs an outside opinion. It gets three, and **they are not equally
trustworthy** -- which is the part worth reading.

- **vs random** saturates almost immediately and then tells you nothing.
- **vs depth-3 exact search** is a real if modest bar.
- **mirror_draw_rate** -- the agent against itself, greedy, noise off -- is the
  one to trust on a board whose proven value is a draw. An agent approaching
  perfect play draws every game, so this climbs from 0 toward 1, and nothing
  about it can be faked by a network that has learned to output zero.

That last point is a direct criticism of the obvious metric. The value head's
opinion of the empty board averages over self-play in which both sides are still
imperfect, so on a drawn board it sits near zero -- which is also exactly what
an untrained `tanh` head emits. It cannot tell a converged agent from an
ignorant one.

The `opening_plies` argument is a bug fix with a story. The first version played
the mirror match greedily from move one, which made all 32 "games" the same
deterministic game repeated 32 times, so the rate could only ever be exactly 0.0
or exactly 1.0. It reported 1.0, 0.0, 1.0, 0.0, 0.0 across a run -- a coin flip
dressed as a measurement -- and that pattern is what gave it away.


In [ ]:
"""Measuring whether self-play is actually producing a stronger player.

Training loss falls whether or not the agent is learning anything useful, so
the loop needs an outside opinion. Three give it here, and they are not
equally trustworthy -- which is the part worth reading.

  * a random mover, which the agent should beat almost always within a couple
    of iterations, and which stops being informative immediately afterwards;
  * a depth-3 exact search, a real if modest bar;
  * the agent against itself, greedy and with the search noise off.

The third is the one to trust on a board whose proven value is a draw. The
value head's opinion of the empty board is the obvious metric and the weaker
one: it averages over self-play in which both sides are still imperfect, and
on a drawn board "near zero" is also what an untrained tanh head emits, so it
cannot distinguish a converged agent from an ignorant one.
"""

import numpy as np



def _random_action(game, board, rng):
    legal = np.nonzero(game.legal_moves(board))[0]
    return int(rng.choice(legal))


def _agent_actions(game, mcts, boards, simulations, rng):
    policies, _ = mcts.search(boards, simulations, add_noise=False, rng=rng)
    return [int(np.argmax(p)) for p in policies]


def play_match(game, mcts, simulations, opponent, num_games, rng,
               agent_first=True):
    """Play num_games in lockstep and return (wins, draws, losses)."""
    boards = [game.initial_board() for _ in range(num_games)]
    scores = [None] * num_games
    live = list(range(num_games))
    # Ply parity decides whose turn it is; boards are always canonical.
    agent_to_move = agent_first
    ply = 0
    max_plies = game.board_shape[0] * game.board_shape[1] + 2

    while live and ply < max_plies:
        if agent_to_move:
            actions = _agent_actions(
                game, mcts, [boards[i] for i in live], simulations, rng)
        else:
            actions = [opponent(game, boards[i], rng) for i in live]

        still_live = []
        for slot, index in enumerate(live):
            boards[index] = game.apply_move(boards[index], actions[slot])
            result = game.outcome(boards[index])
            if result is None:
                still_live.append(index)
            else:
                # result belongs to the player to move now, who is the one that
                # did not just play.
                mover_score = -result
                scores[index] = mover_score if agent_to_move else -mover_score
        live = still_live
        agent_to_move = not agent_to_move
        ply += 1

    for index in live:
        scores[index] = 0.0

    wins = sum(1 for s in scores if s > 0)
    draws = sum(1 for s in scores if s == 0)
    losses = sum(1 for s in scores if s < 0)
    return wins, draws, losses


def mirror_match(game, mcts, simulations, num_games, rng, opening_plies=4):
    """The agent against itself, with the search noise off.

    On a board whose game-theoretic value is a draw, this is the sharpest
    signal the loop has. A weak agent blunders and produces decisive games; an
    agent approaching perfect play draws with itself every time, so the number
    climbs from 0 toward 1 and nothing about it can be faked by a network that
    has learned to output zero.

    The first few plies are sampled rather than taken greedily. Without that,
    every game in the batch is the same deterministic game played over and
    over, and the draw rate can only ever be exactly 0.0 or exactly 1.0 -- a
    coin flip dressed up as a measurement. The first version of this function
    had that bug and reported 1.0, 0.0, 1.0, 0.0, 0.0 across a run, which is
    what finally gave it away.
    """
    boards = [game.initial_board() for _ in range(num_games)]
    scores = [None] * num_games
    live = list(range(num_games))
    ply = 0
    max_plies = game.board_shape[0] * game.board_shape[1] + 2

    while live and ply < max_plies:
        policies, _ = mcts.search([boards[i] for i in live], simulations,
                                  add_noise=False, rng=rng)
        still_live = []
        for slot, index in enumerate(live):
            policy = policies[slot]
            if ply < opening_plies:
                action = int(rng.choice(len(policy), p=policy))
            else:
                action = int(np.argmax(policy))
            boards[index] = game.apply_move(boards[index], action)
            result = game.outcome(boards[index])
            if result is None:
                still_live.append(index)
            else:
                scores[index] = result
        live = still_live
        ply += 1

    for index in live:
        scores[index] = 0.0
    draws = sum(1 for s in scores if s == 0)
    return draws / len(scores)


def evaluate_agent(game, mcts, simulations, num_games, rng, depth=3):
    """Score the current network against random play, shallow exact search,
    and itself. Returned as a dict the training loop folds into its log."""
    half = max(1, num_games // 2)
    heuristic = disc_difference if game.name.startswith("othello") else None

    def shallow(g, board, r):
        return best_action(g, board, depth, heuristic=heuristic, rng=r)

    report = {}
    for label, opponent in (("random", _random_action), ("depth3", shallow)):
        totals = np.zeros(3, dtype=int)
        # Both colours, because a player that is only good going first is a
        # player that has learned the opening and nothing else.
        for agent_first in (True, False):
            totals += play_match(game, mcts, simulations, opponent, half,
                                 rng, agent_first=agent_first)
        wins, draws, losses = totals
        played = wins + draws + losses
        report[f"vs_{label}"] = f"{wins}/{draws}/{losses}"
        report[f"score_{label}"] = round((wins + 0.5 * draws) / played, 3)

    report["mirror_draw_rate"] = round(
        mirror_match(game, mcts, simulations, max(4, num_games // 4), rng), 3)
    return report


## 10. The loop

One iteration is the whole cycle: play a fleet of games against the current
network, fold them into a replay buffer, take a few hundred gradient steps, and
measure. The loop stops on **wall clock**, not after a fixed number of
iterations, because the budget is the thing this project is about -- the
iteration count is the outcome.

There is deliberately **no arena gate**. AlphaGo Zero (2017) kept a champion
network and promoted a challenger only on a 55% win rate. The generalized
AlphaZero (2018) removed that evaluation step entirely and continuously updated
one network, which is what happens here. Historically accurate, and worth adding
back if you extend this to much longer runs on harder games.


In [ ]:
"""The AlphaZero training loop, on a wall-clock budget.

One iteration is the whole cycle: play a fleet of games against the current
network, fold them into a replay buffer, take a few gradient steps, and
measure whether the agent got any better. The loop stops when the budget runs
out rather than after a fixed number of iterations, because the budget is what
this chapter's project is about -- the number of iterations is the outcome,
not the setting.

    python -m src.part_2_methods.ch08_alphazero.train --budget-seconds 3600

The default board is 6x6 Connect Four, whose game-theoretic value is a proven
draw. That matters for what the loop can honestly show: an agent approaching
perfect play on a drawn board ends up drawing against itself every single
time, and ``mirror_draw_rate`` measures exactly that. Pass ``--cols 7`` for the
standard board, a proven first-player win, or ``--game othello`` for the same
agent on a second domain with nothing else changed.

There is deliberately no arena gate. AlphaGo Zero (2017) kept a champion
network and promoted a challenger only on a 55% win rate; the generalized
AlphaZero (2018) removed that evaluation step entirely and continuously
updated one network, which is what happens here. On much longer runs against
harder games, a gate is worth adding back to prevent regression.
"""

import argparse
import json
import time
from collections import deque
from pathlib import Path
from typing import NamedTuple

import numpy as np
import torch
import torch.nn.functional as F



class RunResult(NamedTuple):
    """What a run leaves behind, for a notebook or a test to pick up.

    ``history`` is the per-iteration log, also written to history.json. The
    network and game come back too because the obvious next thing to do with a
    finished run is measure it, and rebuilding them from the checkpoint to do
    that is a step with nothing to teach.
    """

    history: list
    net: object
    game: object
    device: object
    out_dir: Path


def build_game(name, rows, cols):
    if name == "connect4":
        return ConnectFour(rows or 6, cols or 7)
    if name == "othello":
        return Othello(rows or 6)
    raise ValueError(f"unknown game {name!r}")


def loss_fn(policy_logits, value_pred, target_policy, target_value):
    """AlphaZero's single objective, over a trunk shared by both heads.

    The value head is scored against the game's outcome, the policy head
    against the search's own visit counts -- the search is the teacher, and the
    network is trying to predict what the search would have concluded without
    having to run it. The two terms simply add, and both gradients reach the
    same residual trunk, which is what makes one network cheaper than AlphaGo's
    two rather than merely tidier.
    """
    value_loss = F.mse_loss(value_pred, target_value)
    policy_loss = -(target_policy
                    * F.log_softmax(policy_logits, dim=1)).sum(1).mean()
    return value_loss + policy_loss, value_loss.item(), policy_loss.item()


def train_steps(net, optimizer, buffer, device, batch_size, steps, rng):
    """A fixed number of gradient steps over the replay buffer."""
    net.train()
    totals = np.zeros(3)
    if len(buffer) < batch_size:
        return totals
    for _ in range(steps):
        idx = rng.integers(0, len(buffer), size=batch_size)
        planes = torch.from_numpy(
            np.stack([buffer[i][0] for i in idx])).to(device)
        target_policy = torch.from_numpy(
            np.stack([buffer[i][1] for i in idx])).to(device)
        target_value = torch.from_numpy(
            np.array([buffer[i][2] for i in idx], dtype=np.float32)).to(device)

        policy_logits, value_pred = net(planes)
        loss, value_loss, policy_loss = loss_fn(
            policy_logits, value_pred, target_policy, target_value)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        totals += (loss.item(), value_loss, policy_loss)
    net.eval()
    return totals / steps


def root_value(game, net, device):
    """What the network thinks of the opening position.

    The solved-game tables let this one be marked right or wrong. Read it as a
    sanity check rather than as evidence -- see ``arena.mirror_match`` for why.
    """
    planes = torch.from_numpy(
        game.encode(game.initial_board())[None]).to(device)
    with torch.no_grad():
        _, value = net(planes)
    return float(value.item())


def build_parser():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--game", default="connect4",
                        choices=["connect4", "othello"])
    parser.add_argument("--rows", type=int, default=6)
    parser.add_argument("--cols", type=int, default=6)
    parser.add_argument("--budget-seconds", type=float, default=3600)
    parser.add_argument("--max-iterations", type=int, default=1000)
    parser.add_argument("--games-per-iteration", type=int, default=192)
    parser.add_argument("--simulations", type=int, default=50)
    parser.add_argument("--channels", type=int, default=64)
    parser.add_argument("--blocks", type=int, default=4)
    parser.add_argument("--buffer-size", type=int, default=120000)
    parser.add_argument("--batch-size", type=int, default=512)
    parser.add_argument("--train-steps", type=int, default=200)
    parser.add_argument("--lr", type=float, default=2e-3)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--eval-every", type=int, default=2)
    parser.add_argument("--eval-games", type=int, default=40)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument("--out", default="runs")
    return parser


def main(argv=None):
    args = build_parser().parse_args(argv)

    torch.manual_seed(args.seed)
    rng = np.random.default_rng(args.seed)

    game = build_game(args.game, args.rows, args.cols)
    device = pick_device(args.device)
    net = build(game, args.channels, args.blocks).to(device)
    net.eval()
    optimizer = torch.optim.Adam(net.parameters(), lr=args.lr,
                                 weight_decay=args.weight_decay)
    mcts = BatchedMCTS(game, net, device)
    buffer = deque(maxlen=args.buffer_size)

    out_dir = Path(args.out) / f"{game.name}_{int(time.time())}"
    out_dir.mkdir(parents=True, exist_ok=True)

    solved = game.solved_value()
    print(f"game        {game.name}")
    print(f"device      {device}")
    print(f"parameters  {net.parameter_count():,}")
    print(f"budget      {args.budget_seconds:.0f}s")
    if solved is not None:
        print(f"solved value of the opening position: {solved:+d}")
    print()

    history = []
    started = time.time()
    for iteration in range(1, args.max_iterations + 1):
        elapsed = time.time() - started
        if elapsed >= args.budget_seconds:
            print(f"budget spent after {iteration - 1} iterations")
            break

        t0 = time.time()
        examples, results = play_batch(
            game, mcts, args.games_per_iteration, args.simulations, rng=rng)
        buffer.extend(examples)
        selfplay_time = time.time() - t0

        t0 = time.time()
        losses = train_steps(net, optimizer, buffer, device,
                             args.batch_size, args.train_steps, rng)
        train_time = time.time() - t0

        decisive = sum(1 for r in results if r != 0) / len(results)
        record = {
            "iteration": iteration,
            "elapsed": round(time.time() - started, 1),
            "selfplay_seconds": round(selfplay_time, 1),
            "train_seconds": round(train_time, 1),
            "buffer": len(buffer),
            # On a drawn board this should drift down: fewer games decided by
            # a blunder is what getting better looks like here.
            "decisive_fraction": round(decisive, 3),
            "loss": round(float(losses[0]), 4),
            "value_loss": round(float(losses[1]), 4),
            "policy_loss": round(float(losses[2]), 4),
            "root_value": round(root_value(game, net, device), 3),
        }

        if iteration % args.eval_every == 0:
            record.update(evaluate_agent(
                game, mcts, args.simulations, args.eval_games, rng))

        history.append(record)
        print(" ".join(f"{k}={v}" for k, v in record.items()), flush=True)
        (out_dir / "history.json").write_text(json.dumps(history, indent=2))
        torch.save({"model": net.state_dict(), "args": vars(args)},
                   out_dir / "latest.pt")

    total = time.time() - started
    print(f"\ntotal wall clock {total:.0f}s ({total / 60:.1f} min)")
    print(f"artifacts in {out_dir}")
    return RunResult(history, net, game, device, out_dir)


if __name__ == "__main__":
    main()


In [ ]:
# `main` is rebound by two later cells -- solver.py, alphago.py and
# benchmark.py each define one. Hold on to this one before that happens.
train = main

# 30 minutes. Raise it if your session allows; the 6x6 numbers in the chapter
# come from a full hour. Colab's free GPU runs this at roughly laptop pace,
# because most of the wall clock is Python rather than matrix multiplication.
BUDGET_SECONDS = 1800

argv = ["--budget-seconds", str(BUDGET_SECONDS), "--device", DEVICE,
        "--seed", str(SEED), "--rows", "6", "--cols", "6"]
if FAST:
    argv = ["--budget-seconds", "60", "--device", DEVICE, "--seed", str(SEED),
            "--rows", "4", "--cols", "4", "--games-per-iteration", "8",
            "--simulations", "4", "--train-steps", "2", "--batch-size", "16",
            "--eval-every", "1", "--eval-games", "4", "--max-iterations", "2",
            "--channels", "8", "--blocks", "1"]

result = train(argv)

# The trained network comes back with the log, so everything below measures
# what the loop just produced rather than the random weights above.
history, net, game, device = (result.history, result.net, result.game,
                              result.device)


## 11. The curve

Six measures, six panels. They share no axis and are on different scales, so
one chart with six lines would be unreadable rather than compact.

Read them in this order: `loss` and `value_loss` say the optimizer is working.
`decisive_fraction` should **fall** -- on a proven draw, fewer games settled by
a blunder is what getting better looks like. `root_value` should sit near zero
and is the weakest signal here, for the reason section 9 gives. `score_depth3`
and `mirror_draw_rate` are the two that mean something.


In [ ]:
import matplotlib.pyplot as plt

PANELS = [
    ("loss", "total loss", None),
    ("value_loss", "value loss", None),
    ("decisive_fraction", "decisive games (should fall)", (0, 1)),
    ("root_value", "value of the empty board", (-1, 1)),
    ("score_depth3", "score vs depth-3 search", (0, 1)),
    ("mirror_draw_rate", "draw rate against itself", (0, 1)),
]

fig, axes = plt.subplots(2, 3, figsize=(13, 6.5))
for ax, (key, title, ylim) in zip(axes.ravel(), PANELS):
    xs = [r["iteration"] for r in history if key in r]
    ys = [r[key] for r in history if key in r]
    ax.plot(xs, ys, marker="o", markersize=4, color="#2a78d6")
    ax.set_title(title, fontsize=10, loc="left")
    ax.set_xlabel("iteration", fontsize=9)
    if ylim:
        ax.set_ylim(*ylim)
    if key in ("root_value", "decisive_fraction"):
        ax.axhline(0 if key == "root_value" else 0, color="#adb5bd",
                   linewidth=0.8, linestyle="--")
    ax.grid(color="#e9ecef", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

fig.suptitle(f"{len(history)} iterations in {BUDGET_SECONDS}s "
             f"on {DEVICE}", fontsize=12, x=0.01, ha="left")
fig.tight_layout()
plt.show()


In [ ]:
# The same numbers as a table, which is what you quote in a write-up.
cols = ["iteration", "elapsed", "loss", "value_loss", "decisive_fraction",
        "root_value", "score_depth3", "mirror_draw_rate"]
print("  ".join(f"{c[:9]:>9}" for c in cols))
for record in history:
    print("  ".join(f"{record.get(c, '-'):>9}" for c in cols))


## 12. How good did it actually get?

The training log half-answers this. Two questions it does not settle:

**Does the agent agree with the proven value?** On 6x6 the answer should be "the
opening is a draw", so the value head should report something near zero. Be
careful believing this one -- near zero is also the untrained answer.

**How close to perfect is it really?** Perfect play draws with itself 100% of
the time on this board. Measure the mirror draw rate at two search depths: if
the number is real it will *respond* to more search, and if it is the broken
version from section 9 it will sit at exactly 0 or exactly 1 regardless.


In [ ]:
mcts = BatchedMCTS(game, net, device)

planes = torch.from_numpy(game.encode(game.initial_board())[None]).to(device)
with torch.no_grad():
    _, value = net(planes)
print(f"value of the opening   {float(value.item()):+.3f}")
print(f"proven value           {game.solved_value():+d}")
print()

for simulations in ((4, 8) if FAST else (50, 200)):
    rate = mirror_match(game, mcts, simulations, num_games=8 if FAST else 32,
                        rng=np.random.default_rng(7))
    print(f"{simulations:>3} simulations   draw rate {rate:.3f}   "
          f"(perfect play: 1.000)")


## 13. Watch it search

The charts show that the agent improved. They do not show what it is doing.

`animate_search` replays one move simulation by simulation, so you watch the
visit counts start scattered and concentrate. That concentration **is** the
policy improvement AlphaZero trains on: the search produces a better
distribution than the raw network did, and that distribution becomes the
network's next target. The whole algorithm is that loop, and this is the one
place you can see it happen.


In [ ]:
"""Animations of the search, for the notebook.

Two things are worth watching rather than reading about. The first is one
move's search: the visit counts start scattered and concentrate on a few
columns as simulations accumulate, which is what "the search sharpens the
policy" means in practice -- and that sharpened distribution is exactly what
``selfplay.py`` stores as the policy target. The second is a whole game, with
the search's opinion shown beside every move.

Both return a matplotlib animation. In a notebook:

    from IPython.display import HTML
    HTML(animate_search(game, mcts).to_jshtml())
"""

import matplotlib.pyplot as plt
import numpy as np
from matplotlib import animation

# Two hues, far enough apart to stay distinct under colour-vision deficiency.
FIRST, SECOND = "#2a78d6", "#eb6834"
EMPTY, INK, MUTED, SURFACE = "#e8e8e3", "#0b0b0b", "#8a8a85", "#fcfcfb"


def _draw_board(ax, board, game, title):
    """One position, drawn in absolute terms: first player is always blue.

    The agent's boards are canonical -- the side to move is always +1 -- so
    drawing them directly would swap the colours every ply.
    """
    rows, cols = game.board_shape
    ax.clear()
    ax.set_xlim(-0.6, cols - 0.4)
    ax.set_ylim(-0.6, rows - 0.4)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.axis("off")
    ax.set_title(title, fontsize=10, color=INK, loc="left", pad=10)
    for r in range(rows):
        for c in range(cols):
            value = int(board[r, c])
            colour = EMPTY if value == 0 else (FIRST if value == 1 else SECOND)
            ax.add_patch(plt.Circle((c, r), 0.42, color=colour, zorder=2))


def _style_bars(ax, game, ylabel):
    cols = game.action_size
    ax.set_xlim(-0.7, cols - 0.3)
    ax.set_ylim(0, 1.05)
    ax.set_xticks(range(cols))
    ax.set_xlabel("column", fontsize=9, color=MUTED)
    ax.set_ylabel(ylabel, fontsize=9, color=MUTED)
    ax.tick_params(labelsize=8, colors=MUTED)
    ax.grid(axis="y", color="#e6e6e1", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#d8d8d3")


def animate_search(game, mcts, board=None, simulations=120, interval=60):
    """Watch one move's search concentrate, simulation by simulation."""
    board = game.initial_board() if board is None else board
    trace = []
    mcts.search([board], simulations, add_noise=False, trace=trace)
    frames = [step[0] for step in trace]

    fig, (left, right) = plt.subplots(
        1, 2, figsize=(10, 4.2), dpi=80, facecolor=SURFACE,
        gridspec_kw={"width_ratios": [1, 1.3]})
    for ax in (left, right):
        ax.set_facecolor(SURFACE)

    def render(i):
        counts = frames[i]
        _draw_board(left, board, game, "the position - blue to move")
        right.clear()
        _style_bars(right, game, "share of visits")
        best = int(np.argmax(counts))
        colours = [FIRST if c == best else "#b9d2f0" for c in range(len(counts))]
        right.bar(range(len(counts)), counts, color=colours, width=0.68,
                  zorder=2)
        for c, value in enumerate(counts):
            if value > 0.02:
                right.text(c, value + 0.02, f"{value:.2f}", ha="center",
                           fontsize=8, color=INK)
        right.set_title(f"simulation {i + 1} of {len(frames)}", fontsize=10,
                        color=INK, loc="left", pad=10)
        return []

    fig.suptitle("One move's search", fontsize=12, color=INK, x=0.012,
                 ha="left", y=0.98)
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    plt.close(fig)
    return animation.FuncAnimation(fig, render, frames=len(frames),
                                   interval=interval, blit=False)


def animate_game(game, mcts, simulations=100, interval=700, opponent=None,
                 rng=None):
    """Watch a whole game, with the search's opinion beside every move."""
    rng = rng or np.random.default_rng(0)
    board = game.initial_board()
    positions, policies, chosen = [], [], []
    ply = 0
    limit = game.board_shape[0] * game.board_shape[1] + 2

    while game.outcome(board) is None and ply < limit:
        # Store the position in absolute terms so the colours never swap.
        positions.append(board.copy() if ply % 2 == 0 else -board.copy())
        if opponent is not None and ply % 2 == 1:
            action = opponent(game, board, rng)
            policies.append(None)
        else:
            distribution, _ = mcts.search([board], simulations,
                                          add_noise=False, rng=rng)
            policies.append(distribution[0])
            action = int(np.argmax(distribution[0]))
        chosen.append(action)
        board = game.apply_move(board, action)
        ply += 1

    final = board.copy() if ply % 2 == 0 else -board.copy()
    positions.append(final)
    policies.append(None)
    chosen.append(None)

    result = game.outcome(board)
    verdict = ("a draw, which is correct play on this board" if result == 0
               else f"{'second' if ply % 2 == 0 else 'first'} player wins")

    fig, (left, right) = plt.subplots(
        1, 2, figsize=(10, 4.2), dpi=80, facecolor=SURFACE,
        gridspec_kw={"width_ratios": [1, 1.3]})
    for ax in (left, right):
        ax.set_facecolor(SURFACE)

    def render(i):
        last = i == len(positions) - 1
        mover = "blue" if i % 2 == 0 else "orange"
        caption = verdict if last else f"move {i + 1} - {mover} to play"
        _draw_board(left, positions[i], game, caption)
        right.clear()
        _style_bars(right, game, "share of visits")
        counts = policies[i]
        if counts is None:
            right.set_title("", fontsize=10)
            right.text(0.5, 0.5, "game over" if last else "opponent's move",
                       transform=right.transAxes, ha="center", fontsize=10,
                       color=MUTED)
        else:
            pick = chosen[i]
            colours = [FIRST if c == pick else "#b9d2f0"
                       for c in range(len(counts))]
            right.bar(range(len(counts)), counts, color=colours, width=0.68,
                      zorder=2)
            right.set_title(f"the search played column {pick}", fontsize=10,
                            color=INK, loc="left", pad=10)
        return []

    fig.suptitle("A game, and what the search considered", fontsize=12,
                 color=INK, x=0.012, ha="left", y=0.98)
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    plt.close(fig)
    return animation.FuncAnimation(fig, render, frames=len(positions),
                                   interval=interval, blit=False)


In [ ]:
from IPython.display import HTML

HTML(animate_search(game, mcts, simulations=6 if FAST else 60).to_jshtml())


In [ ]:
# A whole game against a depth-4 exact search, with the agent's opinion
# shown beside every one of its own moves.
HTML(animate_game(
    game, mcts, simulations=8 if FAST else 100,
    opponent=lambda g, b, r: best_action(g, b, 2 if FAST else 4, rng=r),
).to_jshtml())


## 14. The AlphaGo measurement

Everything above is AlphaZero. This section is the AlphaGo half, and it is the
one experiment from it that needs no dataset, no training and no cluster.

Silver and colleagues (2016) report that the 13-layer policy network picks a
move in about **3 milliseconds**, while the linear-softmax rollout policy does it
in about **2 microseconds** -- roughly fifteen hundred times faster, at less than
half the accuracy (24% against 57%). That ratio is the entire reason AlphaGo
mixed a fast random playout into every leaf instead of simply asking the
network, and it is why deleting the rollout had to wait until the value head was
good enough to carry a leaf on its own.

The networks below are the published sizes. They will instantiate and run a
forward pass; they will not train here, and the parameter counts are the
contrast the chapter draws.


In [ ]:
"""The published 19x19 architectures -- read and profiled, not trained.

These are the networks the chapter prints: AlphaGo's separate policy and value
networks from Silver and colleagues (2016), and AlphaGo Zero's merged tower
from Silver and colleagues (2017). They are here to be instantiated,
measured and compared against the project's network, not to be trained.
Training them honestly needs the 30 million position KGS corpus, 50 GPUs for a
week for the value network alone, and in AlphaGo Zero's case thousands of
TPUs. Pretending otherwise is the contradiction the chapter's project exists
to avoid.

What *is* runnable from them is ``benchmark.py``, which times ``PolicyNetwork``
against a fast rollout policy and reproduces the millisecond-versus-microsecond
gap that forced AlphaGo to keep rollouts at all.

    python -m src.part_2_methods.ch08_alphazero.alphago

prints each network's parameter count beside the project's, which is the
comparison worth having in front of you before reading ``network.py``.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class PolicyNetwork(nn.Module):
    """AlphaGo's 13-layer supervised-learning policy network.

    48 feature planes in -- stones, liberties, capture sizes, legality, move
    history -- a 5x5 convolution, eleven 3x3 convolutions at 192 filters, and a
    1x1 head that produces one logit per intersection. Trained on 30 million
    positions from the KGS server, it predicted the human expert's move 57% of
    the time.

    The softmax is inside ``forward`` here because that is how the chapter
    prints it. The project's network returns raw logits instead, so the legal
    mask can be applied before any probability mass is spent.
    """

    def __init__(self, num_input_channels=48, filters=192, hidden_layers=11):
        super().__init__()
        # The first layer uses a larger 5x5 kernel.
        self.conv1 = nn.Conv2d(num_input_channels, filters,
                               kernel_size=5, padding=2)
        # Subsequent layers use 3x3 kernels.
        self.hidden_layers = nn.ModuleList([
            nn.Conv2d(filters, filters, kernel_size=3, padding=1)
            for _ in range(hidden_layers)
        ])
        # The final layer outputs one channel for the 19x19 board.
        self.policy_head = nn.Conv2d(filters, 1, kernel_size=1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        for layer in self.hidden_layers:
            x = F.relu(layer(x))
        x = self.policy_head(x)
        # Flatten the 19x19 output into a vector of 361 logits.
        x = x.view(x.size(0), -1)
        return F.softmax(x, dim=1)


class ValueNetwork(nn.Module):
    """AlphaGo's value network: the same feature extractor, a scalar out.

    Identical in shape to the policy network up to the head, and trained with
    mean squared error against the game's eventual outcome. The weights are
    *not* shared -- that is the whole point of the contrast with
    ``network.AlphaZeroNet``, which merges the two into one trunk.

    Trained first on the same human corpus, it memorised: 0.19 mean squared
    error on the training set against 0.37 on held-out positions, because
    successive positions within one game are almost the same position. The fix
    was 30 million self-play games contributing *one* position each.
    """

    def __init__(self, num_input_channels=48, filters=192, hidden_layers=11,
                 board_size=19):
        super().__init__()
        self.conv1 = nn.Conv2d(num_input_channels, filters,
                               kernel_size=5, padding=2)
        self.hidden_layers = nn.ModuleList([
            nn.Conv2d(filters, filters, kernel_size=3, padding=1)
            for _ in range(hidden_layers)
        ])
        # The value head collapses features for scalar regression.
        self.value_conv = nn.Conv2d(filters, 1, kernel_size=1)
        self.fc1 = nn.Linear(board_size * board_size, 256)
        self.fc2 = nn.Linear(256, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        for layer in self.hidden_layers:
            x = F.relu(layer(x))
        x = F.relu(self.value_conv(x))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        # Tanh squashes the estimate into the [-1, 1] range.
        return torch.tanh(self.fc2(x))


class PublishedResBlock(nn.Module):
    def __init__(self, channels=256):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        # The skip connection allows deep gradient flow.
        x = x + residual
        return F.relu(x)


class PublishedAlphaZeroNet(nn.Module):
    """AlphaGo Zero's network at its published size: 19 blocks, 256 channels.

    Same architecture as ``network.AlphaZeroNet`` -- one residual trunk, a
    policy head and a value head -- with both scale knobs at the paper's
    settings. 362 actions is 361 intersections plus a pass.

    It carries roughly fifty times the project's parameters, which is the
    contrast the chapter draws: the algorithm a laptop runs is the published
    one, and the difference is budget rather than kind.
    """

    def __init__(self, input_channels=17, board_size=19, num_actions=362,
                 channels=256, blocks=19):
        super().__init__()
        self.conv_input = nn.Conv2d(input_channels, channels, 3, padding=1)
        self.bn_input = nn.BatchNorm2d(channels)

        # The shared residual trunk extracts generalized board features.
        self.res_blocks = nn.Sequential(
            *[PublishedResBlock(channels) for _ in range(blocks)])

        # Policy head: collapses channels for action logits.
        self.policy_conv = nn.Conv2d(channels, 2, 1)
        self.policy_bn = nn.BatchNorm2d(2)
        self.policy_fc = nn.Linear(2 * board_size ** 2, num_actions)

        # Value head: collapses channels for scalar state evaluation.
        self.value_conv = nn.Conv2d(channels, 1, 1)
        self.value_bn = nn.BatchNorm2d(1)
        self.value_fc1 = nn.Linear(board_size ** 2, 256)
        self.value_fc2 = nn.Linear(256, 1)

    def forward(self, x):
        x = F.relu(self.bn_input(self.conv_input(x)))
        x = self.res_blocks(x)

        p = F.relu(self.policy_bn(self.policy_conv(x)))
        p = p.view(p.size(0), -1)
        # Raw logits; the softmax is applied during MCTS expansion.
        policy_logits = self.policy_fc(p)

        v = F.relu(self.value_bn(self.value_conv(x)))
        v = v.view(v.size(0), -1)
        v = F.relu(self.value_fc1(v))
        value = torch.tanh(self.value_fc2(v))

        return policy_logits, value


def parameter_counts():
    """Every network in the chapter, measured rather than quoted."""

    project = build(ConnectFour(6, 6), channels=64, blocks=4)
    return [
        ("AlphaGo policy network (19x19, 48 planes, 192 filters)",
         sum(p.numel() for p in PolicyNetwork().parameters())),
        ("AlphaGo value network (same extractor, scalar head)",
         sum(p.numel() for p in ValueNetwork().parameters())),
        ("AlphaGo Zero network (19 blocks x 256 channels)",
         sum(p.numel() for p in PublishedAlphaZeroNet().parameters())),
        ("this chapter's project (6x6, 4 blocks x 64 channels)",
         project.parameter_count()),
    ]


def main():
    rows = parameter_counts()
    width = max(len(label) for label, _ in rows)
    print("Parameter counts, by instantiating each network:\n")
    for label, count in rows:
        print(f"  {label:<{width}}  {count:>12,}")
    published, project = rows[2][1], rows[3][1]
    print(f"\n  the published tower is {published / project:.0f}x the "
          f"project's, at the same architecture.")


if __name__ == "__main__":
    main()


In [ ]:
for label, count in parameter_counts():
    print(f"{label:<55} {count:>12,}")


In [ ]:
"""Measuring the gap that forced AlphaGo to keep a rollout policy at all.

Silver and colleagues (2016) report that the 13-layer policy network picks a
move in about 3 milliseconds, while the linear-softmax rollout policy does it
in about 2 microseconds -- roughly fifteen hundred times faster, at less than
half the accuracy (24% against 57%). That ratio is the whole reason AlphaGo's
leaf evaluation mixes a network prediction with a fast random playout instead
of simply asking the network, and it is why AlphaZero could delete the
rollout only once the value head was good enough to carry a leaf alone.

Nothing here needs training or a dataset, so it runs in seconds on any laptop.
It is the one experiment in the AlphaGo half of the chapter a reader can
actually execute.

    python -m src.part_2_methods.ch08_alphazero.benchmark
    python -m src.part_2_methods.ch08_alphazero.benchmark --device mps
"""

import argparse
import time

import numpy as np
import torch



def timeit(fn, repeats):
    fn()  # warm up caches and lazy kernels
    start = time.perf_counter()
    for _ in range(repeats):
        fn()
    return (time.perf_counter() - start) / repeats


class RolloutPolicy:
    """A linear softmax over a handful of sparse pattern features.

    AlphaGo's rollout policy read small 3x3 stone patterns around each
    candidate move and summed one weight per matching pattern. The shape is
    what matters here: a few table lookups per move and no convolution
    anywhere, which is why it runs three orders of magnitude faster than the
    network it stands in for.
    """

    def __init__(self, num_patterns=8192, num_actions=361,
                 active_per_move=6, seed=0):
        rng = np.random.default_rng(seed)
        self.weights = rng.normal(0, 0.5, size=num_patterns).astype(np.float32)
        # Which patterns each candidate move currently matches. In a real
        # engine this is recomputed incrementally as stones are placed.
        self.active = rng.integers(
            0, num_patterns, size=(num_actions, active_per_move))

    def select(self):
        logits = self.weights[self.active].sum(axis=1)
        logits -= logits.max()
        probabilities = np.exp(logits)
        probabilities /= probabilities.sum()
        return int(probabilities.argmax())


def measure(device_name="cpu", batch=256):
    """Return the three timings, in seconds, plus the network's size."""
    device = torch.device(device_name)
    net = PolicyNetwork().to(device).eval()
    parameters = sum(p.numel() for p in net.parameters())

    single = torch.zeros(1, 48, 19, 19, device=device)
    batched = torch.zeros(batch, 48, 19, 19, device=device)

    with torch.no_grad():
        single_time = timeit(lambda: net(single), 30)
        batch_time = timeit(lambda: net(batched), 5)

    rollout_time = timeit(RolloutPolicy().select, 20000)
    return {
        "parameters": parameters,
        "device": str(device),
        "batch": batch,
        "single_seconds": single_time,
        "batch_seconds": batch_time,
        "rollout_seconds": rollout_time,
    }


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--device", default="cpu",
                        help="cpu, cuda or mps (default: cpu, as published)")
    parser.add_argument("--batch", type=int, default=256)
    args = parser.parse_args(argv)

    m = measure(args.device, args.batch)
    per_position = m["batch_seconds"] / m["batch"]

    print(f"policy network        {m['parameters']:,} parameters "
          f"on {m['device']}")
    print(f"  one position        {m['single_seconds'] * 1e3:8.2f} ms")
    print(f"  batch of {m['batch']:<3}        {m['batch_seconds'] * 1e3:8.2f} ms "
          f"({per_position * 1e6:.0f} us per position)")
    print("rollout policy")
    print(f"  one position        {m['rollout_seconds'] * 1e6:8.2f} us")
    print()
    print(f"network / rollout     "
          f"{m['single_seconds'] / m['rollout_seconds']:8.0f}x")
    print(f"batching wins back    "
          f"{m['single_seconds'] / per_position:8.0f}x")
    print()
    print("Silver and colleagues (2016) report 3 ms against 2 us, a ratio of")
    print("about 1500x. The numbers above are this machine's version of the")
    print("same measurement. The ratio here is smaller in both directions --")
    print("this rollout policy is NumPy rather than tuned C, and the network")
    print("runs on hardware a decade newer -- but the shape survives: a leaf")
    print("costs orders of magnitude more through the network than through a")
    print("playout, which is what AlphaGo's lambda was buying.")
    if m["device"] == "cpu":
        print()
        print("Batching wins nothing on CPU, where one position already")
        print("saturates every core. Re-run with --device cuda or --device mps")
        print("to see the number this project's batched self-play depends on.")
    return m


if __name__ == "__main__":
    main()


In [ ]:
# `main` is benchmark.py's by now -- the cell above rebound it.
benchmark = main
benchmark(["--device", DEVICE, "--batch", "16" if FAST else "256"])


## Where to go next

- **`--cols 7`** is the standard Connect Four board, a proven *first*-player
  win. The convergence signal changes completely: `mirror_draw_rate` stops being
  meaningful and `root_value` should climb positive instead. It is a good
  demonstration of why picking the metric before the board matters.
- **`--game othello`** runs the identical agent on 6x6 Othello, a proven
  *second*-player win, so a trained root value should go **negative**. Nothing
  in the agent changes. That is the claim the chapter makes about chess and
  shogi, in a form you can check.
- **Exercise 8.2's lambda sweep.** AlphaGo's leaf evaluation mixed a value
  prediction with a rollout. AlphaZero's `_descend` uses the value head alone.
  Add a rollout and a mixing weight and find out what it buys on this board --
  the answer on a small board is not the answer on 19x19, which is itself the
  lesson.
- **Exercise 8.3's memorisation trap.** AlphaGo's value network overfitted
  badly -- 0.19 training error against 0.37 on held-out positions -- because
  successive positions within one game are nearly the same position. Train on
  every position from a few games instead of one position from many, and you can
  reproduce the gap in minutes.

The repository has the rest: `make run-ch8-solve` for the oracle,
`make run-ch8-benchmark` for the timing above, and
`python -m src.part_2_methods.ch08_alphazero.inspect_agent runs/<run>/latest.pt --play`
to play the thing you just trained.
